# Session 2 — Simulating Production Data Drift

**Goal:** Create realistic-looking drifted data to trigger the monitoring system.

**The scenario:** Three months after deployment, your client ran two business changes:
1. **Aggressive acquisition campaign**: Signed up many new customers (short tenure)
2. **Fiber optic price increase**: +$10/month for fiber optic subscribers

As a result, the input distribution shifted and the model's accuracy degraded.

In [0]:
%run ../utils/config

In [0]:
%pip install ../bundle/wheels/ --quiet

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F

np.random.seed(42)

# Load original data
df_original = spark.table(f"00_shared.telco_churn").toPandas()
print(f"Original dataset: {len(df_original)} rows")
print(f"Original churn rate: {(df_original['Churn']=='Yes').mean():.1%}")
print(f"Original avg tenure: {df_original['tenure'].mean():.1f} months")
print(f"Original avg MonthlyCharges: ${df_original['MonthlyCharges'].mean():.2f}")

## Drift 1: Tenure Shift

New customer acquisition spike: inject 30% more customers with very short tenure (0-3 months).
This shifts the tenure distribution leftward — the model was trained on a different tenure profile.

In [0]:
# Inject synthetic new customers (short tenure)
new_customer_pool = df_original[df_original['tenure'] <= 3].copy()
n_new = int(len(df_original) * 0.30)
new_customers = new_customer_pool.sample(n=n_new, replace=True, random_state=42)
new_customers['tenure'] = np.random.randint(0, 4, size=n_new)
new_customers['customerID'] = [f"DRIFT-ACQ-{i:04d}" for i in range(n_new)]
new_customers['TotalCharges'] = new_customers['tenure'] * new_customers['MonthlyCharges']

df_drifted = pd.concat([df_original, new_customers], ignore_index=True)
print(f"After acquisition drift: {len(df_drifted)} rows")
print(f"New avg tenure: {df_drifted['tenure'].mean():.1f} months (was {df_original['tenure'].mean():.1f})")

## Drift 2: MonthlyCharges Shift

Fiber optic price increase: add $10 to all fiber optic subscribers.

In [0]:
# Increase MonthlyCharges for fiber optic customers
fiber_mask = df_drifted['InternetService'] == 'Fiber optic'
df_drifted.loc[fiber_mask, 'MonthlyCharges'] += 10.0
df_drifted.loc[fiber_mask, 'TotalCharges'] += (10.0 * df_drifted.loc[fiber_mask, 'tenure'])

print(f"New avg MonthlyCharges: ${df_drifted['MonthlyCharges'].mean():.2f} "
      f"(was ${df_original['MonthlyCharges'].mean():.2f})")

## Drift 3: Label Drift

The combination of new customers + higher prices causes more people to churn.
Flip 25% of high-risk 'No' customers to 'Yes'.

In [0]:
# Flip some high-risk non-churners to churners
high_risk_mask = (
    (df_drifted['Contract'] == 'Month-to-month') &
    (df_drifted['MonthlyCharges'] > 70) &
    (df_drifted['Churn'] == 'No')
)
flip_candidates = df_drifted[high_risk_mask]
flip_idx = flip_candidates.sample(frac=0.25, random_state=42).index
df_drifted.loc[flip_idx, 'Churn'] = 'Yes'

print(f"Original churn rate : {(df_original['Churn']=='Yes').mean():.1%}")
print(f"Drifted churn rate  : {(df_drifted['Churn']=='Yes').mean():.1%}")

## Write the Combined Training Dataset

Now that we have a drifted dataset with ground-truth labels, we can create a richer training
set for the retrain pipeline. Rather than replacing the original data, we **append** the
drifted rows to the original — this is the standard continual-learning approach:

- The model doesn't "forget" the original distribution
- It gains new examples that reflect the current production environment
- More total samples generally improve generalisation

The retrain job will pick this table up automatically when triggered from <span style="color:cc0000;background-color:#e0e0e0;font-family:courier">05_trigger_retrain.ipynb</span>.

In [0]:
# Combine original data with the drifted dataset (union, not replace)
# df_drifted has the same schema as telco_churn — no extra columns to drop
df_combined = pd.concat([df_original, df_drifted], ignore_index=True)
df_combined['TotalCharges'] = pd.to_numeric(df_combined['TotalCharges'], errors='coerce')

# Write to participant's personal schema
training_table = "telco_churn_new_training_data"
spark.createDataFrame(df_combined).write.mode("overwrite").saveAsTable(training_table)

print(f"✓ Wrote {len(df_combined):,} rows to {training_table}")
print(f"  Original rows  : {len(df_original):,}")
print(f"  Drifted rows   : {len(df_drifted):,}")
print(f"  Combined churn rate : {(df_combined['Churn']=='Yes').mean():.1%}  (original was {(df_original['Churn']=='Yes').mean():.1%})")

## Write the Inference Log

Simulate what an inference log looks like: original features + model predictions.
We'll add some noise to the predictions (the model is making more mistakes on this new distribution).

In [0]:
import mlflow

mlflow.set_registry_uri("databricks-uc")

# Load the champion model so we can simulate inference predictions on drifted data coming from this model
#    The model you trained in your personal schema
model_name   = f"{catalog}.{schema}.churn_classifier"
champion_uri = f"models:/{model_name}@champion"

# Extract the model ID with a fallback value, if needed
try:
    model = mlflow.sklearn.load_model(champion_uri)
    from mlflow import MlflowClient
    client = MlflowClient()
    mv = client.get_model_version_by_alias(model_name, "champion")
    model_id = f"{model_name}_v{mv.version}"
except Exception as e:
    print(f"Could not load champion model: {e}")
    print("Using 'mock_model_v1' as model_id instead")
    model = None
    model_id = "mock_model_v1"

# Read the connfig file
import yaml
config_path = "../common/config.yml"
with open(config_path) as f:
    config = yaml.safe_load(f)

# Prepare training dataframes for the drifted data
from churn_model.features import prepare_dataframe
X_drifted, y_drifted = prepare_dataframe(df_drifted, config)

# Run predictions on drifted data using the current champion model
if model is not None:
    predictions = model.predict(X_drifted)
else:
    # Mock predictions with intentional degradation
    predictions = (np.random.rand(len(df_drifted)) > 0.65).astype(int)

df_log = df_drifted.copy()

# Prepare predicted log data
#     Ensure TotalCharges is numeric (telco_churn has 11 empty-string rows that force STRING type)
df_log['TotalCharges'] = pd.to_numeric(df_log['TotalCharges'], errors='coerce')
df_log['prediction']  = ['Yes' if p == 1 else 'No' for p in predictions]
df_log['model_id']    = model_id
df_log['inference_ts'] = pd.Timestamp.utcnow()

# Write to inference log table
inference_table = f"churn_inference_log"
spark.createDataFrame(df_log).write.mode("overwrite").saveAsTable(inference_table)
spark.sql(f"ALTER TABLE {inference_table} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

print(f"✓ Wrote {len(df_log):,} rows to {inference_table}")
print(f"\nDrifted predictions:")
print(f"  Predicted churn rate : {(df_log['prediction']=='Yes').mean():.1%}")
print(f"  Actual churn rate    : {(df_log['Churn']=='Yes').mean():.1%}")

## Summary

We've created an inference log with 3 types of drift:

| Type | Feature | Before | After | Expected PSI |
|---|---|---|---|---|
| Covariate | `tenure` | ~32 months | ~22 months | ~0.28 |
| Covariate | `MonthlyCharges` | ~$64 | ~$68 | ~0.12 |
| Label | `Churn` rate | ~26% | ~32% | - |



➡️ Next: [03_monitoring_setup.ipynb]($./03_monitoring_setup) — set up Lakehouse Monitoring to detect this drift